In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

df = pd.read_csv('fmcg_sales_3years_1M_rows.csv')
df.head(5)

In [ ]:
def country_sales():
    sales_by_country = df.groupby('country')['net_sales'].sum().reset_index()
    sales_by_country = sales_by_country.set_index('country')
    return sales_by_country

print(country_sales().sort_values(by='net_sales', ascending=False))

In [ ]:
#Ranking SKUs by revenues (cumulative gross sales)
skus = df.groupby(['sku_id', 'sku_name'])['gross_sales'].sum().reset_index()
skus = skus.set_index('sku_id')
sku_revenues = skus.sort_values(by='gross_sales', ascending=False)
print("Total SKUs:", sku_revenues['sku_name'].count())
print(sku_revenues.head(20))

In [ ]:
A_threshold = 0.80
B_threshold = 0.95

sku_revenues['cumulative_value'] = sku_revenues['gross_sales'].cumsum()
total_value = sku_revenues['gross_sales'].sum()

sku_revenues['cumulative_pct'] = sku_revenues['cumulative_value'] / total_value

# Class assignment
sku_revenues['class'] = 'C'
sku_revenues.loc[sku_revenues['cumulative_pct'] <= A_threshold, 'class'] = 'A'
sku_revenues.loc[
    (sku_revenues['cumulative_pct'] > A_threshold) &
    (sku_revenues['cumulative_pct'] <= B_threshold),
    'class'
] = 'B'

sku_revenues.reset_index()
sku_revenues.to_csv('abc_sku_classification.csv', index=False)

abc_sku = df.merge(sku_revenues[['sku_id', 'class']], on='sku_id', how='left')

summary = abc_sku.groupby(['country', 'class']).agg(
    num_skus=('sku_id', 'nunique'),
    total_value=('gross_sales', 'sum')
).reset_index()

print(summary.pivot_table(index='country', columns='class', values=['num_skus', 'total_value'], aggfunc='sum'))

In [ ]:
sku_stats = df.groupby(['sku_id', 'sku_name'])['gross_sales'].agg(mean='mean', std_dev='std').reset_index()
sku_stats['mean'] = sku_stats['mean'].round(2)
sku_stats['std_dev'] = sku_stats['std_dev'].round(2)
sku_stats['cv'] = sku_stats['std_dev'] / sku_stats['mean']
sku_stats['cv'] = sku_stats['cv'].round(2)

x_threshold = 0.20
y_threshold = 0.50

def assign_volatility(cv):
    if cv <= x_threshold:
        return 'X'
    elif cv <= y_threshold:
        return 'Y'
    else:
        return 'Z'

sku_stats['xyz'] = sku_stats['cv'].apply(assign_volatility)
sku_stats.to_csv('sku_xyz_analysis_value.csv', index=False)

In [ ]:
#Find correlation between list_price, units_sold, and promo_flag
correlation_matrix = df[['list_price', 'units_sold', 'promo_flag']].corr()
print(correlation_matrix)

In [ ]:
'''Weekly seasonality
Grouping by week and using avg for each week over the 3 years.

'''
weekly_data = df.groupby('weekofyear').agg(
    rain_mm=('rain_mm', 'mean'),
    temperature=('temperature', 'mean')
).reset_index()

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(weekly_data['weekofyear'], weekly_data['rain_mm'], label='Rainfall (mm)', color='r', marker='o', linewidth=2)
ax.set_xlabel('Week of Year')
ax.set_ylabel('Average Rainfall (mm)', color='r')
ax.tick_params(axis='y', labelcolor='r')

ax2 = ax.twinx()
ax2.plot(weekly_data['weekofyear'], weekly_data['temperature'], label='Temperature (C)', color='b', marker='^', linewidth=2)
ax2.set_ylabel('Average Temperature (C)', color='b')
ax2.tick_params(axis='y', labelcolor='b')

plt.title('Weekly Weather Seasonality (Averaged over 3 Years)')
plt.xticks(range(1, 54, 4))  # Show every 4th week for readability
plt.grid(True, alpha=0.3)
plt.legend(loc='upper right')
plt.tight_layout()
plt.show()

In [ ]:
#Seasonality monthly
monthly_data = df.groupby('month').agg(
    rain_mm=('rain_mm', 'mean'),
    temperature=('temperature', 'mean')
).reset_index()

fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(monthly_data['month'], monthly_data['rain_mm'], label='Rainfall (mm)', color='skyblue')
ax.set_xlabel('Month')
ax.set_ylabel('Average Rainfall (mm)', color='r')

ax2 = ax.twinx()
ax2.plot(monthly_data['month'], monthly_data['temperature'], label='Temperature (C)', color='b', marker='^')
ax2.set_ylabel('Average Temperature (C)', color='b')

plt.xticks(range(1, 13, 1))
plt.title('Weather Seasonality')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
#Seasonality yearly
yearly_data = df.groupby('year').agg(
    rain_mm=('rain_mm', 'mean'),
    temperature=('temperature', 'mean')
).reset_index()

fig, ax = plt.subplots(figsize=(8, 6))
ax.bar(yearly_data['year'], yearly_data['rain_mm'], label='Rainfall (mm)', color='skyblue')
ax.set_xlabel('Year')
ax.set_ylabel('Average Rainfall (mm)')

ax2 = ax.twinx()
ax2.plot(yearly_data['year'], yearly_data['temperature'], label='Temperature (C)', color='b', marker='^')
ax2.set_ylabel('Average Temperature (C)', color='b')

plt.tick_params(axis='y', labelcolor='b')
plt.title('Annual Weather Seasonality')
plt.xticks(rotation=90)
plt.show()